I've wanted to create a RAG chatbot using LlamaIndex and the QWEN LLM model. The multilingual e5-large model was selected as an embedding model. As a reranker, I chose ms-marco-MiniLM-L4-v2, good enough for small RAG bots. All models were chosen for efficiency, speed, and good quality. Unfortunately, I can't demonstrate the real questions because of the secure policy, but I added two questions to show that the RAG system really works and could be implemented in the workflow. Of course, any model could be easily replaced. For example, I have tried Llama-3.2-3B-Instruct, Qwen2.5-3B-Instruct, and faced the problem that small models with 3B params usually make up questions and answer them, even if you don't ask them about it. Qwen3-4B-Instruct-2507 model shows the perfect tradeoff between good quality (especially for the full Russian questions and document) and the possibility to run the model on a Google Colab. The reranker helps the system produce more accurate answers; without it, the model more frequently retrieved irrelevant documents and produced incorrect responses.
Running on Google Colab with a T4 GPU, end-to-end inference latency is around 30-60 seconds, reflecting a quality–performance trade-off for a RAG pipeline over an 8000-word document.   Additionally, it is possible to connect it to WhatsApp, Telegram, or any other chat apps using APIs.

In [ ]:
pip install llama_index python-docx llama_index.llms.huggingface llama_index.embeddings.huggingface llama-index-readers-markitdown llama-index-postprocessor-rankllm-rerank huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# data preprocessing. Splitting raw document into categories and sections.
# I had a 41-page DOCX file with the company’s internal reward policy, which I automated by creating a RAG-based chatbot.
# I couldn’t share the file, and the name was hidden for security reasons.
from docx import Document

doc = Document("/content/drive/MyDrive/***.docx")

sections = []
current_section = ""

filtered_text = ''
category = ''
for para in doc.paragraphs:
    if para.style.name.startswith("Heading"):
        category = para.text.strip()
    else:
        text_by_numbers = []
        if para.style.name.startswith("List Paragraph"):
            filtered_text += '\n\n' + category + '+,+' + para.text.strip() + '\n'
        elif para.style.name.startswith("Normal"):
            if len(para.text.strip()) != 0:
                filtered_text += para.text.strip() + '\n'

In [5]:
total_text_by_numbers = []

for i in filtered_text.split('\n\n'):
    if len(i.strip()) != 0:
        total_text_by_numbers.append((i.strip().split('+,+')[0].strip(), i.strip().split('+,+')[1].strip()))

In [ ]:
# Normalization pretty boosts the overall model answer's quality.

import re
import unicodedata

def normalize_text(text: str) -> str:
    text = text.lower()

    text = ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c))

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r"[^\w\s\d\-.,]", " ", text)
    text = re.sub(r'[“”«»]', '', text)
    text = re.sub(r'[_*]', '', text)

    text = text.replace('–', '-').replace('—', '-')

    return text

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.schema import TextNode
from huggingface_hub import login
from google.colab import userdata

nodes = []

for category, answer in total_text_by_numbers:
    embedding_text = f"категория: {normalize_text(category)}\n{normalize_text(answer)}"

    node = TextNode(
        text=answer,
        embedding_text=embedding_text,
        metadata={"category": category})

    nodes.append(node)

embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large")

llm = HuggingFaceLLM(
    model_name="Qwen/Qwen3-4B-Instruct-2507",
    tokenizer_name="Qwen/Qwen3-4B-Instruct-2507",
    context_window=4096,
    max_new_tokens=100,
    device_map="auto",
    generate_kwargs = {"temperature": 0.001,
                       "top_p": 0.2
                      })

index = VectorStoreIndex(nodes,embed_model=embed_model)

In [8]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L4-v2",
    top_n=1
)

In [9]:
from llama_index.core import PromptTemplate

template = PromptTemplate(
    "You are a helpful assistant. Answer the question ONLY using the provided retrieved documents!"
    "Answer ONLY in Russian language!"
    "Always answer with full text provided in the retrieved documents, do not cut it!"
    "Do NOT add extra commentary, explanations, or prompts for the next question!"
    "Do NOT use any outside knowledge! Do NOT make up answers!"
    "Never apply the identity rule unless the user explicitly asks about identity.\n\n"
    "If the answer cannot be found in the documents, reply exactly: 'Извините, не смог найти информацию. Попробуйте перефразировать свой вопрос.'\n\n"

    "Retrieved documents:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)


query_engine = index.as_query_engine(llm=llm, text_qa_template=template,  similarity_top_k=5,
    node_postprocessors=[reranker],
    response_mode="compact")

In [ ]:
# This actually was necessary for smaller models, when they usually add this remark even if the answer was correctly found.

def response(input_text):
    response_text = str(query_engine.query(input_text).response)
    if response_text.strip() == 'Извините, не смог найти информацию. Попробуйте перефразировать свой вопрос.':
        return 'Извините, не смог найти информацию. Попробуйте перефразировать свой вопрос.'
    else:
        return response_text.replace('Извините, не смог найти информацию. Попробуйте перефразировать свой вопрос.', '').strip()

In [11]:
response("Что такое ПДЗ?")

'ПДЗ – Просроченная дебиторская задолженность.'

In [12]:
print(response('Кто такой Нурсултан Назарбаев?'))

Извините, не смог найти информацию. Попробуйте перефразировать свой вопрос.
